# Parsing Propositional Logic

F# counterpart of `Python/Chapter-10/Propositional-Logic.ipynb`. Run this notebook from top to bottom with a fresh .NET (F#) kernel.  The scanner and the parser are generated by the F# port of `Larkita` from the grammar `logic.lark`.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open LarkitaAst

## Grammar and typed syntax

The grammar encodes the precedences structurally:  equivalence is non-associative, since its two operands are
implications; implication associates to the right, disjunction and conjunction to the left, and negation binds most
tightly.  The string literals of an alternative form the head of the abstract syntax tree, e.g. `p ∧ q` yields
`('∧', ('ID', 'p'), ('ID', 'q'))`, while `⊤` yields `('⊤',)`.  The function `toExpression` converts these trees into
the closed type `Expression`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "logic.lark")

In [ ]:
/// A closed syntax tree makes every interpreter case visible to the compiler.
type Expression =
    | Number of float
    | Name of string
    | Unary of string * Expression
    | Binary of string * Expression * Expression

type Statement =
    | Assign of string * Expression
    | Evaluate of Expression

In [ ]:
let logicParser = withConflictWarnings Raise (fun () -> Larkita.Open("logic.lark", outFile = "logic.out"))

let rec toExpression (t: Ast) : Expression =
    match t with
    | Tok("ID", name) -> Name name
    | Node([ ("⊤" | "⊥") as constant ], []) -> Name constant
    | Node([ "¬" ], [ operand ]) -> Unary("¬", toExpression operand)
    | Node([ operator ], [ left; right ]) -> Binary(operator, toExpression left, toExpression right)
    | _ -> failwithf "Unexpected tuple %s" (toString t)

module Parse =
    let logic text = toExpression (parse logicParser text)

The output file `logic.out` describes the states of the canonical LR(1) automaton.  The grammar has no conflicts.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "logic.out")

In [ ]:
#load "../Support/Visualization.fsx"
open Visualization

let rec toTree expression =
    match expression with
    | Number value -> Leaf (string value)
    | Name name -> Leaf name
    | Unary (operator, operand) -> Visualization.Node (operator, [toTree operand])
    | Binary (operator, left, right) -> Visualization.Node (operator, [toTree left; toTree right])

let test parse text =
    let expression = parse text
    printfn "%s ⇒ %A" text expression
    expression

Equivalence is non-associative: its two operands are implications. This grammar encodes that constraint structurally, Implication associates to the right, disjunction and conjunction to the left, and negation binds most tightly.

In [ ]:
let examples =
    [ "(p ↔ q) ∨ (q ↔ p) → (¬q ↔ ¬p) ∧ (q → ⊥ ↔ p → ⊥)"
      "p ∧ ⊤"; "q ∨ ⊥"; "¬p ∧ q"; "p ∧ q → r"; "p → q → r"
      "¬(p ∨ q) ↔ ¬p ∧ ¬q"; "¬¬p ↔ p"
      "(p → q) ∧ (q → r) → p → r"; "p ∨ q ∧ r"
      "v ↔ u → p ∨ q ∧ ¬ ⊥" ]
let trees = examples |> List.map (test Parse.logic)
checkNotebook (Parse.logic "p → q → r" = Binary ("→", Name "p", Binary ("→", Name "q", Name "r")))
checkNotebook (Parse.logic "p ∨ q ∧ r" = Binary ("∨", Name "p", Binary ("∧", Name "q", Name "r")))
let nonAssociative = try Parse.logic "p ↔ q ↔ r" |> ignore; false with _ -> true
checkNotebook nonAssociative

## Evaluate a formula

A valuation maps proposition names to truth values. The compiler checks that every syntax-tree constructor is handled. Missing propositions raise an explicit error.

In [ ]:
let rec evaluate values formula =
    match formula with
    | Name "⊤" -> true
    | Name "⊥" -> false
    | Name name -> Map.find name values
    | Unary ("¬", operand) -> not (evaluate values operand)
    | Binary (operator, left, right) ->
        let p, q = evaluate values left, evaluate values right
        match operator with
        | "∧" -> p && q
        | "∨" -> p || q
        | "→" -> not p || q
        | "↔" -> p = q
        | _ -> invalidArg "formula" operator
    | other -> invalidArg "formula" (sprintf "Not a logical formula: %A" other)

let tautology = Parse.logic "¬(p ∨ q) ↔ ¬p ∧ ¬q"
for p in [false; true] do
    for q in [false; true] do
        checkNotebook (evaluate (Map.ofList ["p", p; "q", q]) tautology)

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(trees[6] |> toTree |> treeToDot |> renderDot), "text/html") |> ignore